# 🚀 AGAR-RL V10 : MaskablePPO SOTA (Action Masking, LayerNorm Trunk & Cosine Annealing LR)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/train_colab.ipynb)

**Entraînement de haute performance à pleine puissance (GPU L4 / A100) avec auto-sauvegarde Google Drive.**

### 🎯 Nouveautés Majeures de la V10 (Architecture Moderne Deep RL 2026) :
1. **Action Masking Strict (`MaskablePPO` via `sb3-contrib`)** : Verrouillage mathématique à $-\infty$ du logit de split quand l'action est physiquement impossible (masse < 36 ou déjà 16 sous-cellules). Éradication absolue de 100% des splits invalides sans friction artificielle.
2. **Tronc & Têtes Normalisées (`LayerNorm(512)` / `RMSNorm(512)`)** : Stabilisation profonde des représentations latentes de l'extracteur et des têtes Policy/Value face à la dérive de distribution en Self-Play multi-agents.
3. **Planification Cosine Annealing du Learning Rate** : Décroissance douce et progressive du taux d'apprentissage de $3 \times 10^{-4} \to 1 \times 10^{-5}$ pour consolider les réflexes de chasse et affiner la précision de visée des splits.
4. **Récompenses SOTA V9 Conservées** : Maintien des victoires de la V9 (Kills proportionnels à la masse dévorée, delta de masse pur, zéro split-for-food).
5. **Sauvegarde Continue & Reprise Hiérarchique V10** : Dossier Google Drive `/content/drive/MyDrive/agario_rl_backup_v10`, avec reprise automatique sur V10 prioritaire, puis V9, V8, V7, V6 ou V5 !
6. **Export & Replay HD V10** : Replay vidéo 80s `eval_match_v10.mp4` et export standard `model_v10.onnx`.

## 0. Montage Google Drive & Détection GPU L4
Tous les checkpoints, les replays HD et les modèles ONNX seront automatiquement sauvegardés sur votre Drive dans le dossier `agario_rl_backup_v8`.

In [ ]:
# 1. Montage sécurisé de Google Drive
import os, sys, time, torch

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup_v10'
PREV_BACKUP_V9 = '/content/drive/MyDrive/agario_rl_backup_v9'
PREV_BACKUP_V8 = '/content/drive/MyDrive/agario_rl_backup_v8'
PREV_BACKUP_V7 = '/content/drive/MyDrive/agario_rl_backup_v7'
PREV_BACKUP_V6 = '/content/drive/MyDrive/agario_rl_backup_v6'
PREV_BACKUP_V5 = '/content/drive/MyDrive/agario_rl_backup_v5'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# 2. Vérification du matériel accéléré (GPU L4 / A100 recommandé)
print('=' * 65)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'🚀 Accélération GPU Détectée : {gpu_name} ({vram:.1f} Go VRAM)')
    print('⚡ Configuration optimale : 16 environnements parallèles + MaskablePPO + Numba JIT + Batch 1024')
else:
    print('⚠️ Aucun GPU détecté. Activez un GPU dans : Exécution > Modifier le type d\'exécution')
print(f'📁 Dossier Google Drive V10 synchronisé : {DRIVE_BACKUP_DIR}')
print(f'📁 Dossier V9 précédent disponible : {PREV_BACKUP_V9} (Existe: {os.path.exists(PREV_BACKUP_V9)})')
print(f'📁 Dossier V8 précédent disponible : {PREV_BACKUP_V8} (Existe: {os.path.exists(PREV_BACKUP_V8)})')
print('=' * 65)


## 1. Synchronisation du Code GitHub & Installation des Dépendances

In [ ]:
import os

# 1. Récupération propre des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Synchronisation avec GitHub main...')
    !git fetch origin main
    !git reset --hard origin/main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et synchronisation avec GitHub main...')
    %cd agario
    !git fetch origin main
    !git reset --hard origin/main
else:
    print('🌐 Clonage propre du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances Farama Gymnasium
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt tensorboard
!apt-get install -qq -y ffmpeg
print('✅ Environnement et dépendances installés avec succès.')

## 2. Validation Pré-Vol : Suite Complète de 34 Tests Unitaires
Vérification complète de la physique du moteur, du remerge magnétique, des récompenses de traque et de l'espace d'observation log-ratio.

In [ ]:
# Exécute tous les tests du moteur physique, du remerge et des récompenses Farama
!python -m pytest -v

## 4. Entraînement Haute Performance V10 (MaskablePPO + LayerNorm + Action Masking)
- **Action Masking Strict** : Empêche structurellement l'agent de tenter des splits illégaux.
- **Sauvegarde Continue V10** : Checkpoints automatiques tous les 250 000 pas dans `agario_rl_backup_v10`.

In [ ]:
# 🚀 Configuration de Reprise & Lancement V10
import os, glob, re

V10_DIR = '/content/drive/MyDrive/agario_rl_backup_v10'
V9_DIR = '/content/drive/MyDrive/agario_rl_backup_v9'
V8_DIR = '/content/drive/MyDrive/agario_rl_backup_v8'
V7_DIR = '/content/drive/MyDrive/agario_rl_backup_v7'
V6_DIR = '/content/drive/MyDrive/agario_rl_backup_v6'
V5_DIR = '/content/drive/MyDrive/agario_rl_backup_v5'
os.makedirs(V10_DIR, exist_ok=True)

def extract_step(path):
    fname = os.path.basename(path)
    if 'final' in fname:
        return 999_999_999
    m = re.search(r'step_(\d+)', fname)
    return int(m.group(1)) if m else 0

def find_best_checkpoint(dir_path):
    if not os.path.exists(dir_path):
        return None
    zips = glob.glob(os.path.join(dir_path, '*.zip'))
    valid = [z for z in zips if os.path.getsize(z) > 1000 and not os.path.basename(z).startswith('._') and 'bc_pretrained' not in z]
    if not valid:
        return None
    valid.sort(key=extract_step, reverse=True)
    return valid[0]

# Recherche hiérarchique : V10 en premier, puis V9, V8, V7, V6, V5
chosen_checkpoint = (
    find_best_checkpoint(V10_DIR) or
    find_best_checkpoint(V9_DIR) or
    find_best_checkpoint(V8_DIR) or
    find_best_checkpoint(V7_DIR) or
    find_best_checkpoint(V6_DIR) or
    find_best_checkpoint(V5_DIR)
)

if chosen_checkpoint:
    resume_flag = f'--resume "{chosen_checkpoint}"'
    step_num = extract_step(chosen_checkpoint)
    print('=' * 75)
    print(f'🎯 Checkpoint source détecté pour reprise : {chosen_checkpoint}')
    print(f'📊 Palier détecté : {step_num:,} steps' if step_num < 999_999_999 else '📊 Palier : FINAL')
    print('=' * 75)
else:
    resume_flag = '--resume auto'
    print('⚠️ Aucun checkpoint préalable trouvé, démarrage d\'un nouvel entraînement.')

!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 20000000 \
    --pool-interval 250000 \
    --backup-dir {V10_DIR} \
    {resume_flag} \
    --device auto


## 4. Entraînement Haute Performance V8 (Authentic Competitive Agar.io)
- **Reprise Optimale depuis V7 / V6** : Détection automatique du plus récent checkpoint dans `agario_rl_backup_v7` (ou `v6`) pour continuer l'entraînement avec la physique authentique des 16 cellules et la perception sous-cellulaire !
- **Sauvegarde Continue V8** : Checkpoints automatiques tous les 250 000 pas dans `agario_rl_backup_v8`.

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

def get_best_in_dir(dir_path):
    if not os.path.exists(dir_path):
        return None
    zips = glob.glob(os.path.join(dir_path, '*.zip'))
    valid = [z for z in zips if os.path.getsize(z) > 1000 and not os.path.basename(z).startswith('._') and 'bc_pretrained' not in z]
    if not valid:
        return None
    valid.sort(key=extract_step, reverse=True)
    return valid[0]

search_dirs = [
    '/content/drive/MyDrive/agario_rl_backup_v10',
    '/content/drive/MyDrive/agario_rl_backup_v9',
    '/content/drive/MyDrive/agario_rl_backup_v8',
    '/content/drive/MyDrive/agario_rl_backup_v7',
    '/content/drive/MyDrive/agario_rl_backup_v6',
    'checkpoints/ppo'
]

target_inspect = None
for sd in search_dirs:
    target_inspect = get_best_in_dir(sd)
    if target_inspect:
        break
target_inspect = target_inspect or 'checkpoints/ppo/ppo_latest.zip'

print('=' * 75)
print(f'🔬 Inspection Diagnostique du Modèle : {target_inspect}')
print('=' * 75)

!python src/analysis/inspect_policy.py --model "{target_inspect}"


## 5. Inspection Diagnostique de la Politique & Réflexes Tactiques
Sonde le réseau de neurones sur des scénarios synthétiques contrôlés : réponse à la nourriture, esquive des prédateurs mortels vs calme face aux rivaux inoffensifs, et propension à attaquer les proies en zone de frappe.

In [ ]:
import os, glob, re
from IPython.display import HTML, display
from base64 import b64encode

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

def get_best_in_dir(dir_path):
    if not os.path.exists(dir_path):
        return None
    zips = glob.glob(os.path.join(dir_path, '*.zip'))
    valid = [z for z in zips if os.path.getsize(z) > 1000 and not os.path.basename(z).startswith('._') and 'bc_pretrained' not in z]
    if not valid:
        return None
    valid.sort(key=extract_step, reverse=True)
    return valid[0]

search_dirs = [
    '/content/drive/MyDrive/agario_rl_backup_v10',
    '/content/drive/MyDrive/agario_rl_backup_v9',
    '/content/drive/MyDrive/agario_rl_backup_v8',
    '/content/drive/MyDrive/agario_rl_backup_v7',
    '/content/drive/MyDrive/agario_rl_backup_v6',
    'checkpoints/ppo'
]

target_model = None
for sd in search_dirs:
    target_model = get_best_in_dir(sd)
    if target_model:
        break
target_model = target_model or 'checkpoints/ppo/ppo_latest.zip'
step_count = extract_step(target_model)

print('=' * 75)
print(f'🎬 Modèle sélectionné pour le Replay HD : {target_model}')
print(f'📊 Palier : {step_count:,} steps' if step_count < 999_999_999 else '📊 Palier : FINAL')
print('=' * 75)

os.makedirs('recordings', exist_ok=True)
!python src/inference/record_match.py \
    --model "{target_model}" \
    --output recordings/eval_match_v10.mp4 \
    --steps 2400

if os.path.exists('recordings/eval_match_v10.mp4') and os.path.exists('/content/drive/MyDrive/agario_rl_backup_v10'):
    !cp recordings/eval_match_v10.mp4 /content/drive/MyDrive/agario_rl_backup_v10/eval_match_v10.mp4
    print('📁 Replay HD copié sur Google Drive dans : agario_rl_backup_v10/eval_match_v10.mp4')

video_path = 'recordings/eval_match_v10.mp4'
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="850" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(video_path) / 1_000_000:.1f} Mo')
else:
    print('⚠️ Vidéo non trouvée.')


## 6. Enregistrement Automatique du Match Replay HD & Visualisation Directe
Génère une vidéo HD de 80 secondes (2400 steps @ 30 FPS) avec affichage tête haute (HUD), vecteurs de décision et radar.

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

def get_best_in_dir(dir_path):
    if not os.path.exists(dir_path):
        return None
    zips = glob.glob(os.path.join(dir_path, '*.zip'))
    valid = [z for z in zips if os.path.getsize(z) > 1000 and not os.path.basename(z).startswith('._') and 'bc_pretrained' not in z]
    if not valid:
        return None
    valid.sort(key=extract_step, reverse=True)
    return valid[0]

search_dirs = [
    '/content/drive/MyDrive/agario_rl_backup_v10',
    '/content/drive/MyDrive/agario_rl_backup_v9',
    '/content/drive/MyDrive/agario_rl_backup_v8',
    '/content/drive/MyDrive/agario_rl_backup_v7',
    '/content/drive/MyDrive/agario_rl_backup_v6',
    'checkpoints/ppo'
]

best_model = None
for sd in search_dirs:
    best_model = get_best_in_dir(sd)
    if best_model:
        break

if best_model and os.path.exists(best_model):
    print(f'Modèle sélectionné pour l\'export : {best_model}')
    os.makedirs('models', exist_ok=True)
    !python src/inference/export_onnx.py --model "{best_model}" --output models/model_v10.onnx
    if os.path.exists('/content/drive/MyDrive/agario_rl_backup_v10'):
        !cp models/model_v10.onnx /content/drive/MyDrive/agario_rl_backup_v10/model_v10.onnx
        print('📁 Modèle ONNX sauvegardé sur Drive : agario_rl_backup_v10/model_v10.onnx')
else:
    print('⚠️ Aucun checkpoint trouvé pour l\'export ONNX.')


## 7. Exportation Universelle vers ONNX & Benchmark de Latence
Convertit le réseau de neurones PyTorch au standard ONNX ultra-rapide (< 0.02 ms de latence CPU).

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

def get_best_in_dir(dir_path):
    if not os.path.exists(dir_path):
        return None
    zips = glob.glob(os.path.join(dir_path, '*.zip'))
    valid = [z for z in zips if os.path.getsize(z) > 1000 and not os.path.basename(z).startswith('._') and 'bc_pretrained' not in z]
    if not valid:
        return None
    valid.sort(key=extract_step, reverse=True)
    return valid[0]

search_dirs = [
    '/content/drive/MyDrive/agario_rl_backup_v9',
    '/content/drive/MyDrive/agario_rl_backup_v8',
    '/content/drive/MyDrive/agario_rl_backup_v7',
    '/content/drive/MyDrive/agario_rl_backup_v6',
    'checkpoints/ppo'
]

best_model = None
for sd in search_dirs:
    best_model = get_best_in_dir(sd)
    if best_model:
        break

if best_model and os.path.exists(best_model):
    print(f'Modèle sélectionné pour l\'export : {best_model}')
    os.makedirs('models', exist_ok=True)
    !python src/inference/export_onnx.py --model "{best_model}" --output models/model_v9.onnx
    if os.path.exists('/content/drive/MyDrive/agario_rl_backup_v9'):
        !cp models/model_v9.onnx /content/drive/MyDrive/agario_rl_backup_v9/model_v9.onnx
        print('📁 Modèle ONNX sauvegardé sur Drive : agario_rl_backup_v9/model_v9.onnx')
else:
    print('⚠️ Aucun checkpoint trouvé pour l\'export ONNX.')
